# 18 Compute stock risk and liquidity features

目标：计算 10 个风险、规模、换手和流动性特征，Beta 使用日度 log 超额收益和重叠三日收益。

- 输入：14 号月度/日度股票及 FF 因子
- 正式输出：stock_risk_liquidity_features_raw.parquet、观察数审计
- 主键：permno+month
- 上游依赖：14
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import duckdb
import yaml
from src.atomic_write import duckdb_copy_parquet_atomic
from src.deterministic_risk import (
    SUV_COLUMN_NAME,
    write_deterministic_daily_month_features,
    write_monthly_oos_suv_features,
)
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
monthly_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_monthly_clean.parquet")
daily_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/stock_daily_clean.parquet")
ffm_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/ff_monthly_clean.parquet")
ffd_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/ff_daily_clean.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=true")
con.execute(f"""CREATE TEMP VIEW daily_base AS
 SELECT d.*,f.mktrf,f.smb,f.hml,f.rf,
   date_trunc('month',d."date")::DATE month_value,
   year(d."date")*12+month(d."date") month_index,
   CASE WHEN d.dlyret>-1 AND f.rf>-1
     THEN ln(1+d.dlyret)-ln(1+f.rf) END stock_log_excess,
   CASE WHEN f.mktrf+f.rf>-1 AND f.rf>-1
     THEN ln(1+f.mktrf+f.rf)-ln(1+f.rf) END market_log_excess,
   CASE WHEN d.dlyret IS NOT NULL AND f.rf IS NOT NULL
     THEN d.dlyret-f.rf END simple_excess
 FROM read_parquet('{esc(daily_path)}') d
 LEFT JOIN read_parquet('{esc(ffd_path)}') f ON d."date"=f."date" """)
con.execute("""CREATE TEMP VIEW daily_three AS
 SELECT *,
   CASE WHEN count(stock_log_excess) OVER w3=3
     THEN sum(stock_log_excess) OVER w3 END stock_3d,
   CASE WHEN count(market_log_excess) OVER w3=3
     THEN sum(market_log_excess) OVER w3 END market_3d
 FROM daily_base
 WINDOW w3 AS (PARTITION BY permno ORDER BY "date" ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)""")
vol_days=int(PARAMS["beta_vol_window_days"])
corr_days=int(PARAMS["beta_corr_window_days"])
con.execute(f"""CREATE TEMP VIEW daily_rolling AS
 SELECT *,
   count(CASE WHEN stock_log_excess IS NOT NULL AND market_log_excess IS NOT NULL
     THEN 1 END) OVER wvol n_beta_vol,
   stddev_samp(stock_log_excess) OVER wvol stock_sd_1y,
   stddev_samp(market_log_excess) OVER wvol market_sd_1y,
   count(CASE WHEN stock_3d IS NOT NULL AND market_3d IS NOT NULL
     THEN 1 END) OVER wcorr n_beta_corr,
   corr(stock_3d,market_3d) OVER wcorr corr_3d_5y,
   max(abs(dlyprc)) OVER wvol year_high,
   row_number() OVER(PARTITION BY permno,month_value ORDER BY "date" DESC) month_last_rank
 FROM daily_three
 WINDOW
   wvol AS (PARTITION BY permno ORDER BY "date" ROWS BETWEEN {vol_days-1} PRECEDING AND CURRENT ROW),
   wcorr AS (PARTITION BY permno ORDER BY "date" ROWS BETWEEN {corr_days-1} PRECEDING AND CURRENT ROW)""")
beta_stage=PATHS["full_run_tmp"]/"18_beta_monthly_staged.parquet"
stats_stage=PATHS["full_run_tmp"]/"18_daily_month_stats_staged.parquet"
suv_stage=PATHS["full_run_tmp"]/"18_monthly_oos_suv_v1_staged.parquet"
final_stage=PATHS["full_run_tmp"]/"18_risk_features_staged.parquet"
for p in [beta_stage,stats_stage,suv_stage,final_stage]: p.unlink(missing_ok=True)
duckdb_copy_parquet_atomic(con,f"""SELECT permno,month_value AS "month",
 CASE WHEN n_beta_vol>={int(PARAMS["beta_vol_min_obs"])}
        AND n_beta_corr>={int(PARAMS["beta_corr_min_obs"])}
        AND market_sd_1y>0
   THEN corr_3d_5y*stock_sd_1y/market_sd_1y END "Beta",
 year_high,n_beta_vol,n_beta_corr
 FROM daily_rolling WHERE month_last_rank=1""",beta_stage)
con.execute(f"CREATE TEMP VIEW beta_monthly AS SELECT * FROM read_parquet('{esc(beta_stage)}')")
deterministic_daily_audit=write_deterministic_daily_month_features(
    con,
    source_relation="daily_base",
    output_path=stats_stage,
    idio_min_obs=int(PARAMS["idio_vol_min_obs"]),
    resid_var_min_obs=int(PARAMS["resid_var_min_obs"]),
    variance_min_obs=int(PARAMS["variance_min_obs"]),
    suv_min_obs=int(PARAMS["suv_min_obs"]),
)
suv_config=yaml.safe_load((ROOT/"config"/"suv_monthly_oos_v1.yaml").read_text(encoding="utf-8"))
assert suv_config["method"]=="monthly_oos_suv_v1"
assert int(suv_config["estimation_window"])==60
assert suv_config["window_semantics"]=="last_valid_stock_observations"
assert int(suv_config["min_estimation_obs"])==40
assert int(suv_config["min_evaluation_obs"])==10
assert int(suv_config["gap_trading_days"])==0
assert suv_config["monthly_aggregation"]=="mean"
assert float(suv_config["sigma_floor"])==1e-12
assert suv_config["exact_cpz_replication"] is False
assert suv_config["implementation_family"]=="Garfinkel-style out-of-sample abnormal volume"
assert suv_config["legacy_method"]["name"]=="legacy_same_sample_zero"
assert suv_config["legacy_method"]["deprecated"] is True
assert suv_config["legacy_method"]["formal_production_method"] is False
suv_oos_audit=write_monthly_oos_suv_features(
    con,
    source_relation="daily_base",
    output_path=suv_stage,
    estimation_window=int(suv_config["estimation_window"]),
    min_estimation_obs=int(suv_config["min_estimation_obs"]),
    min_evaluation_obs=int(suv_config["min_evaluation_obs"]),
    gap_trading_days=int(suv_config["gap_trading_days"]),
    sigma_floor=float(suv_config["sigma_floor"]),
)
con.execute(f"""CREATE TEMP VIEW daily_month_features AS
 SELECT * FROM read_parquet('{esc(stats_stage)}')""")
con.execute(f"""CREATE TEMP VIEW monthly_oos_suv AS
 SELECT * FROM read_parquet('{esc(suv_stage)}')""")
con.execute(f"""CREATE TEMP VIEW monthly_base AS
 SELECT m.*,f.mktrf,f.rf,year(m."month")*12+month(m."month") month_index,
   CASE WHEN m.mthret IS NOT NULL AND f.rf IS NOT NULL AND f.mktrf IS NOT NULL
     THEN m.mthret-f.rf END stock_excess
 FROM read_parquet('{esc(monthly_path)}') m
 LEFT JOIN read_parquet('{esc(ffm_path)}') f ON m."month"=f."date" """)
mkt_window=int(PARAMS["mkt_beta_window_months"])
con.execute(f"""CREATE TEMP VIEW monthly_beta AS
 SELECT *,
   count(CASE WHEN stock_excess IS NOT NULL AND mktrf IS NOT NULL THEN 1 END)
     OVER w n_mktbeta,
   regr_slope(stock_excess,mktrf) OVER w "MktBeta"
 FROM monthly_base
 WINDOW w AS (PARTITION BY permno ORDER BY month_index
   RANGE BETWEEN {mkt_window-1} PRECEDING AND CURRENT ROW)""")
names=['Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
stock_feature_names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
assert stock_feature_names[44]==SUV_COLUMN_NAME
suv_column_index=stock_feature_names.index(SUV_COLUMN_NAME)
assert suv_column_index==44
feature_query=f"""SELECT m.permno,m."month",
 b."Beta",s."IdioVol",m.market_equity_thousand "LME",
 CASE WHEN m.shrout>0 THEN m.mthvol/(m.shrout*1000) END "LTurnover",
 CASE WHEN m.n_mktbeta>={int(PARAMS["mkt_beta_min_obs"])} THEN m."MktBeta" END "MktBeta",
 CASE WHEN b.year_high>0 THEN abs(m.mthprc)/b.year_high END "Rel2High",
 s."Resid_Var",s."Spread",o."SUV",s."Variance"
 FROM monthly_beta m
 LEFT JOIN beta_monthly b USING(permno,"month")
 LEFT JOIN daily_month_features s USING(permno,"month")
 LEFT JOIN monthly_oos_suv o USING(permno,"month")"""
out=output_path(ROOT,RUN_MODE,"18_compute_stock_risk_liquidity_features","data/processed/stock/stock_risk_liquidity_features_raw.parquet")
aud=output_path(ROOT,RUN_MODE,"18_compute_stock_risk_liquidity_features","data/audit/18_risk_feature_observation_counts.parquet")
duckdb_copy_parquet_atomic(con,feature_query,final_stage)
duckdb_copy_parquet_atomic(con,f"""SELECT m.permno,m."month",
 b.n_beta_vol,b.n_beta_corr,s.n_ff3,s.n_ff3_2,o.n_est AS n_suv,
 o.n_eval,o.sigma_hat,o.missing_reason AS suv_missing_reason,
 s.n_ret_2,s.n_spread,m.n_mktbeta
 FROM monthly_beta m
 LEFT JOIN beta_monthly b USING(permno,"month")
 LEFT JOIN daily_month_features s USING(permno,"month")
 LEFT JOIN monthly_oos_suv o USING(permno,"month")""",aud)
con.execute(f"CREATE TEMP VIEW features AS SELECT * FROM read_parquet('{esc(final_stage)}')")
monthly_count=int(con.execute("SELECT count(*) FROM monthly_base").fetchone()[0])
daily_count=int(con.execute("SELECT count(*) FROM daily_base").fetchone()[0])
feature_count=int(con.execute("SELECT count(*) FROM features").fetchone()[0])
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM features GROUP BY permno,"month" HAVING count(*)>1)""").fetchone()[0])
feature_columns=[r[0] for r in con.execute("DESCRIBE features").fetchall()]
assert feature_columns==["permno","month"]+names
assert duplicate_count==0
con.close()
out.parent.mkdir(parents=True,exist_ok=True)
os.replace(final_stage,out)
beta_stage.unlink(missing_ok=True); stats_stage.unlink(missing_ok=True); suv_stage.unlink(missing_ok=True)
INPUTS_USED=["stock_monthly_clean.parquet","stock_daily_clean.parquet","ff_monthly_clean.parquet","ff_daily_clean.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud]]
INPUT_COUNTS={"monthly":monthly_count,"daily":daily_count}; OUTPUT_COUNTS={"features":feature_count}; PRIMARY_KEY=["permno","month"]; DUPLICATE_KEY_COUNT=duplicate_count
EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=str(aud.relative_to(ROOT)); STAGE_STATUS="passed"
STAGE_NOTES=["Beta uses stock and market daily log excess returns and overlapping three-day sums; MKT_RF is not reduced by RF a second time.","IdioVol and Resid_Var retain stable permno/date ordering, float64 explicit-driver least squares and compensated RSS.","SUV is monthly_oos_suv_v1: the last 60 valid observations strictly before the evaluation month estimate a float64 intercept/r_plus/r_minus model; at least 40 estimation and 10 evaluation observations are required, and historical sigma uses n_est-3 degrees of freedom.","legacy_same_sample_zero remains deprecated and is not selected by the production SUV configuration.","Spread and Variance use stable two-pass or compensated reductions; critical linear algebra is single-threaded and no raw-moment subtraction formula is used.","Feature month m uses information ending no later than m and is paired with m+1 return only in the direction estimator."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "18_compute_stock_risk_liquidity_features", stage="18", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
